# Vision app

In [1]:
%matplotlib widget
from dorna_vision import Detection_app
x = Detection_app()

# Eye in hand calibration

In [5]:
from dorna2 import Dorna
from camera import Camera
from dorna_vision import Detection
import time
import json
import config
#####################
import config

index = 0
pxl_list = [[378, 190]]
pxl = pxl_list[index]
rvec_base = [180, 0, 0]
ip = config.robot["ip"]
joint_img = config.tube_pick["joint"]+config.tube_pick["aux"]
tool = config.two_finger_gripper["15ml_cap"]["tool"]
preset = {
        "camera_mount":{
            "type": "dorna_ta_j4",
            "ej": [0 ,0, 0, 0, 0, 0, 0, 0],
            "T": [46.5174596+1+1+0+4-(1), 32.0776662-3+1-0-1.5+(-1), -4.24772615-3, -0.27547989, 0.27691881, 89.6939516],
        },
}
sim=0
speed = 0.05
###### initialization
# robot
robot = Dorna()
robot.connect(ip)

# camera
camera = Camera()
camera.connect()
# detection
d = Detection(robot=robot, camera=camera, **preset)
######
# go imaging
robot.go(joint=joint_img)
time.sleep(0.5)

# run detection
d.run()

# get xyz
xyz = d.xyz(pxl)
xyz[2] +=2

# go xyz
tvec = xyz+rvec_base
retval = robot.go(motion="lmove", pose=tvec, tool=tool, speed=speed, sim=sim)
for r in retval:
    print(json.dumps(r))

robot.close()
camera.close()
d.close()

{"cmd": "lmove", "rel": 0, "vel": 40.0, "accel": 300.0, "jerk": 500.0, "cont": 0, "corner": 50, "j0": 89.95568418888041, "j1": 33.350250774566994, "j2": -97.8462682245563, "j3": -2.5311202512043565, "j4": -26.963538015858944, "j5": 93.46937577497151}


# Calibration with joints

In [2]:
from dorna2 import Dorna
from dorna2 import pose as dorna_pose
import numpy as np
import config

# rail config
frame_in_world = config.robot["frame_in_world"]
base_in_world = config.robot["base_in_world"]
aux_dir = config.robot["aux_dir"]

# measured
#m_joint_rec = {"cmd":"jmove","rel":0,"j0":50.053711,"j1":35.81543,"j2":-102.260742,"j3":-0.505371,"j4":-23.554688,"j5":-39.396973}
m_joint_rec = {"cmd":"jmove","rel":0,"j0":54.360352,"j1":35.969238,"j2":-108.654785,"j3":-0.373535,"j4":-17.336426,"j5":-35.15625}

m_aux = config.decapper["aux"]

# target[
target_aux = config.decapper["aux"]
target_frame = config.decapper["frame"]
tool = config.two_finger_gripper["15ml_cap"]["tool"]
target_pose = config.decapper["15ml_cap"]

####### measured
m_joint = [m_joint_rec[k] for k in ["j"+str(i) for i in range(6)]]
robot = Dorna()
robot.kinematic.set_tcp_xyzabc(tool)
robot.kinematic.fw(m_joint)
m_xyzabc = robot.kinematic.fw(m_joint)
m_xyzabc_world = dorna_pose.robot_to_frame(robot.kinematic.fw(m_joint), aux=m_aux, aux_dir=aux_dir, base_in_world=base_in_world, frame_in_world = frame_in_world)
print(m_xyzabc_world)

####### object
target_xyzabc_world = dorna_pose.transform_pose(target_pose, from_frame=target_frame, to_frame=[0,0,0,0,0,0])
print(target_xyzabc_world)

####### recalculate o_frame
# ── 2) Build 4×4 transforms for world and local poses ────────────────────
T_world = np.array(dorna_pose.xyzabc_to_T(m_xyzabc_world))
T_local = np.array(dorna_pose.xyzabc_to_T(target_pose))

# ── 3) Solve for the object frame:  T_frame = T_world @ inv(T_local) ─────
T_frame = T_world @ np.linalg.inv(T_local)
target_frame_np = dorna_pose.T_to_xyzabc(T_frame)
target_frame_new = [float(x) for x in target_frame_np]
print("####### target_frame_new ###########")
print(target_frame_new)

#### recalculate target_pose
target_pose_new = list(map( float, dorna_pose.transform_pose( m_xyzabc_world, from_frame=[0, 0, 0, 0, 0, 0], to_frame=target_frame)))
print("####### target_pose_new ###########")
print(target_pose_new)

[334.92870741429607, -39.902221169718956, 122.10110400292716, 179.9218956618693, -0.1991073692976828, -0.129292853239163]
[337.12171872025147, -45.49057287490242, 121.0497545666377, 179.9273520662806, -0.1262469938012151, -0.08304432450854272]
####### target_frame_new ###########
[309.7755063457461, -31.259957029685296, 2.125313751147658, 179.921895632902, -0.19910736926562048, -0.12929285321837136]
####### target_pose_new ###########
[22.798176783140775, 3.1660670419544203, -121.0563977696934, -0.005362523643205572, -0.029482697114558285, 0.0463906712373938]


In [ ]:
[309.760356693039, -30.550253000508803, 1.0747388584599378, 179.8675734256945, -0.13748847103630463, -0.3009983336471652]
[309.411317891139, -29.71600544282014, 1.0192087461651056, 179.95060766362727, -0.12877701091867336, -0.18172538979393776]
[309.7755063457461, -31.259957029685296, 2.125313751147658, 179.921895632902, -0.19910736926562048, -0.12929285321837136]


# Rotation

In [3]:
from dorna2 import pose

abc = [0, 0, 0]

abc = pose.rotate_abc(abc, axis=[1, 0, 0], angle=180, local=True)
abc = pose.rotate_abc(abc, axis=[0, 0, 1], angle=45, local=True)
abc

[166.2983158520316, -68.88301782571617, 4.217729064991604e-15]

# Go

In [2]:
from dorna2 import Dorna
import time
freedom = {"num":50, "range":[0.5, 0.5, 0.5], "early_exit":True}
robot = Dorna()
current_joint =  [80.112305, 68.862305, -117.685547, -0.021973, -41.176758, 35.134277, 388.74375]
pose =  [-300.4584516808835, 186.73230510526315, 250.1425720414547, -159.00911604446466, 65.86373240657448, -23.19692136401907, 390.35]

final_joint = [141.745605, 48.88916, -80.002441, -10.700684, -72.355957, -168.815918, 390.35625]


#robot.kinematic.inv(pose[0:6], current_joint[0:6], False, freedom=freedom)
robot.go(pose=pose, current_joint=current_joint, motion="lmove", freedom= freedom, cont=0, corner=50, timeout=-1, sim=1)   


[{'cmd': 'lmove',
  'rel': 0,
  'accel': 1200.0,
  'jerk': 2000.0,
  'cont': 0,
  'corner': 50,
  'j0': 141.7349335654358,
  'j1': 48.88754038426882,
  'j2': -79.98482008833054,
  'j3': -10.698901419291417,
  'j4': -72.36859331669143,
  'j5': -168.8100032051365,
  'j6': 390.35}]